# EDA y Quality Control - Precipitación mensual

Este notebook realiza la **Fase 2** del flujo de trabajo de AgroPilot: exploración, diagnóstico y control de calidad de la base mensual de precipitación.

Insumos esperados:

- `data/processed/precipitacion_mensual.csv`
- `data/processed/series_precipitacion.csv`
- `data/spatial/E_precipitacion.gpkg`

Salidas principales:

- `data/processed/qc_precipitacion_mensual.csv`
- `data/processed/qc_resumen_estaciones.csv`
- `data/processed/qc_estaciones_precipitacion.gpkg`

Objetivo: identificar cobertura temporal, vacíos, duplicados, valores negativos, posibles atípicos y condiciones generales de calidad antes de pasar a interpolación o modelación.

## 1. Librerías y rutas

Se definen las rutas según la arquitectura actual del proyecto. Este notebook está pensado para ubicarse en:

`notebooks/exploration/EDA_precipitacion.ipynb`

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import geopandas as gpd
    GEOPANDAS_DISPONIBLE = True
except ImportError:
    GEOPANDAS_DISPONIBLE = False

BASE_DIR = Path.cwd()

RUTA_PRECIPITACION = Path("../../data/processed/precipitacion_mensual.csv")
RUTA_SERIES = Path("../../data/processed/series_precipitacion.csv")
RUTA_ESTACIONES_GPKG = Path("../../data/spatial/E_precipitacion.gpkg")

RUTA_SALIDA_QC_MENSUAL = Path("../../data/processed/qc_precipitacion_mensual.csv")
RUTA_SALIDA_QC_RESUMEN = Path("../../data/processed/qc_resumen_estaciones.csv")
RUTA_SALIDA_QC_GPKG = Path("../../data/processed/qc_estaciones_precipitacion.gpkg")

for ruta in [RUTA_PRECIPITACION, RUTA_SERIES, RUTA_ESTACIONES_GPKG]:
    print(ruta.resolve(), "→", ruta.exists())

## 2. Carga de datos

Se cargan las observaciones mensuales, el resumen de series y la capa espacial de estaciones.

In [ ]:
precipitacion = pd.read_csv(RUTA_PRECIPITACION)
series = pd.read_csv(RUTA_SERIES)

precipitacion["CodigoEstacion"] = precipitacion["CodigoEstacion"].astype(str)
series["CodigoEstacion"] = series["CodigoEstacion"].astype(str)

precipitacion["Fecha"] = pd.to_datetime(precipitacion["Fecha"], errors="coerce")
precipitacion["Valor"] = pd.to_numeric(precipitacion["Valor"], errors="coerce")

series["FechaInicio"] = pd.to_datetime(series["FechaInicio"], errors="coerce")
series["FechaFin"] = pd.to_datetime(series["FechaFin"], errors="coerce")

if GEOPANDAS_DISPONIBLE:
    estaciones = gpd.read_file(RUTA_ESTACIONES_GPKG)
    estaciones["CodigoEstacion"] = estaciones["CodigoEstacion"].astype(str)
else:
    estaciones = None
    print("GeoPandas no está disponible. Se omite la carga espacial.")

print("Precipitación mensual:", precipitacion.shape)
print("Series:", series.shape)
if estaciones is not None:
    print("Estaciones espaciales:", estaciones.shape)

## 3. Vista inicial de las bases

Se revisa estructura, tipos de datos y una muestra de registros.

In [ ]:
display(precipitacion.head())
display(precipitacion.info())
display(series.head())

if estaciones is not None:
    display(estaciones.head())
    print(estaciones.crs)

## 4. Diagnóstico general

Conteo de estaciones, rango temporal global y magnitud total de registros.

In [ ]:
diagnostico_general = pd.DataFrame({
    "Indicador": [
        "Registros totales",
        "Estaciones únicas",
        "Fecha mínima global",
        "Fecha máxima global",
        "Valores nulos en Fecha",
        "Valores nulos en Valor"
    ],
    "Valor": [
        len(precipitacion),
        precipitacion["CodigoEstacion"].nunique(),
        precipitacion["Fecha"].min(),
        precipitacion["Fecha"].max(),
        precipitacion["Fecha"].isna().sum(),
        precipitacion["Valor"].isna().sum()
    ]
})

diagnostico_general

## 5. Duplicados por estación y mes

Un duplicado ocurre cuando una misma estación tiene más de un registro para el mismo mes.

In [ ]:
precipitacion["Periodo"] = precipitacion["Fecha"].dt.to_period("M")

duplicados = (
    precipitacion
    .groupby(["CodigoEstacion", "Periodo"])
    .size()
    .reset_index(name="Conteo")
    .query("Conteo > 1")
    .sort_values(["CodigoEstacion", "Periodo"])
)

print("Duplicados estación-mes:", len(duplicados))
display(duplicados.head(20))

## 6. Vacíos temporales por estación

Se calcula, para cada estación, cuántos meses deberían existir entre su primera y última fecha, cuántos están disponibles y cuántos faltan.

In [ ]:
def resumen_vacios_estacion(grupo):
    codigo = grupo.name
    fechas = grupo["Fecha"].dropna().sort_values()

    if fechas.empty:
        return pd.Series({
            "FechaInicio_QC": pd.NaT,
            "FechaFin_QC": pd.NaT,
            "MesesEsperados_QC": 0,
            "MesesObservados_QC": 0,
            "MesesFaltantes_QC": 0,
            "Completitud_QC": np.nan,
            "BrechaMaximaMeses": np.nan
        })

    periodo_inicio = fechas.min().to_period("M")
    periodo_fin = fechas.max().to_period("M")
    periodos_esperados = pd.period_range(periodo_inicio, periodo_fin, freq="M")
    periodos_observados = fechas.dt.to_period("M").unique()
    faltantes = periodos_esperados.difference(periodos_observados)

    if len(faltantes) == 0:
        brecha_maxima = 0
    else:
        faltantes_ordenados = pd.Series(faltantes.astype(str)).sort_values()
        faltantes_dt = pd.to_datetime(faltantes_ordenados.astype(str) + "-01")
        cortes = faltantes_dt.diff().dt.days.ne(31).cumsum()
        brecha_maxima = faltantes_dt.groupby(cortes).size().max()

    return pd.Series({
        "FechaInicio_QC": fechas.min(),
        "FechaFin_QC": fechas.max(),
        "MesesEsperados_QC": len(periodos_esperados),
        "MesesObservados_QC": len(periodos_observados),
        "MesesFaltantes_QC": len(faltantes),
        "Completitud_QC": len(periodos_observados) / len(periodos_esperados),
        "BrechaMaximaMeses": brecha_maxima
    })

resumen_vacios = (
    precipitacion
    .groupby("CodigoEstacion")
    .apply(resumen_vacios_estacion)
    .reset_index()
)

resumen_vacios.sort_values("Completitud_QC").head(20)

## 7. Valores negativos y valores nulos

En precipitación mensual, un valor negativo debe marcarse como inconsistente.

In [ ]:
precipitacion["flag_valor_nulo"] = precipitacion["Valor"].isna()
precipitacion["flag_valor_negativo"] = precipitacion["Valor"] < 0

resumen_basico_qc = (
    precipitacion
    .groupby("CodigoEstacion")
    .agg(
        Registros=("Valor", "size"),
        ValoresNulos=("flag_valor_nulo", "sum"),
        ValoresNegativos=("flag_valor_negativo", "sum"),
        ValorMinimo=("Valor", "min"),
        ValorMaximo=("Valor", "max"),
        ValorMedio=("Valor", "mean")
    )
    .reset_index()
)

resumen_basico_qc.sort_values(["ValoresNegativos", "ValoresNulos"], ascending=False).head(20)

## 8. Detección preliminar de valores atípicos

Se usa una regla robusta basada en el rango intercuartílico por estación y mes calendario.

No se eliminan datos. Solamente se marcan para revisión.

In [ ]:
precipitacion["Mes"] = precipitacion["Fecha"].dt.month

estadisticos_mes = (
    precipitacion
    .groupby(["CodigoEstacion", "Mes"])["Valor"]
    .agg(
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        mediana="median",
        conteo="count"
    )
    .reset_index()
)

estadisticos_mes["iqr"] = estadisticos_mes["q3"] - estadisticos_mes["q1"]
estadisticos_mes["limite_inferior"] = estadisticos_mes["q1"] - 1.5 * estadisticos_mes["iqr"]
estadisticos_mes["limite_superior"] = estadisticos_mes["q3"] + 1.5 * estadisticos_mes["iqr"]

precipitacion = precipitacion.merge(
    estadisticos_mes[["CodigoEstacion", "Mes", "limite_inferior", "limite_superior", "mediana", "conteo"]],
    on=["CodigoEstacion", "Mes"],
    how="left"
)

precipitacion["flag_atipico_iqr"] = (
    (precipitacion["conteo"] >= 10) &
    (
        (precipitacion["Valor"] < precipitacion["limite_inferior"]) |
        (precipitacion["Valor"] > precipitacion["limite_superior"])
    )
)

atipicos = precipitacion[precipitacion["flag_atipico_iqr"]].copy()
print("Registros marcados como posibles atípicos:", len(atipicos))
display(atipicos[["CodigoEstacion", "NombreEstacion", "Fecha", "Valor", "limite_inferior", "limite_superior"]].head(30))

## 9. Resumen QC por estación

Se consolida un resumen por estación con vacíos, nulos, negativos, duplicados y posibles atípicos.

In [ ]:
resumen_duplicados = (
    duplicados
    .groupby("CodigoEstacion")
    .agg(MesesDuplicados=("Periodo", "count"))
    .reset_index()
)

resumen_atipicos = (
    precipitacion
    .groupby("CodigoEstacion")
    .agg(PosiblesAtipicos=("flag_atipico_iqr", "sum"))
    .reset_index()
)

qc_resumen = (
    resumen_vacios
    .merge(resumen_basico_qc, on="CodigoEstacion", how="left")
    .merge(resumen_duplicados, on="CodigoEstacion", how="left")
    .merge(resumen_atipicos, on="CodigoEstacion", how="left")
)

qc_resumen["MesesDuplicados"] = qc_resumen["MesesDuplicados"].fillna(0).astype(int)
qc_resumen["PosiblesAtipicos"] = qc_resumen["PosiblesAtipicos"].fillna(0).astype(int)

qc_resumen["CalidadSerie"] = np.select(
    [
        qc_resumen["Completitud_QC"] >= 0.90,
        qc_resumen["Completitud_QC"] >= 0.70,
        qc_resumen["Completitud_QC"] < 0.70
    ],
    ["Alta", "Media", "Baja"],
    default="Sin clasificar"
)

qc_resumen = qc_resumen.sort_values(["CalidadSerie", "Completitud_QC"], ascending=[True, False])
qc_resumen.head(20)

## 10. Integración con la capa espacial

Se une el resumen QC con `E_precipitacion.gpkg`, conservando geometría, altitud y atributos espaciales.

In [ ]:
if estaciones is not None:
    estaciones_qc = estaciones.merge(
        qc_resumen,
        on="CodigoEstacion",
        how="left"
    )
    display(estaciones_qc.head())
else:
    estaciones_qc = None
    print("No se generó capa espacial QC porque GeoPandas no está disponible.")

## 11. Visualizaciones exploratorias

Estas gráficas ayudan a entender cobertura, completitud, distribución de datos y comportamiento mensual.

In [ ]:
plt.figure(figsize=(10, 5))
qc_resumen["Completitud_QC"].dropna().hist(bins=20)
plt.title("Distribución de completitud por estación")
plt.xlabel("Completitud")
plt.ylabel("Número de estaciones")
plt.show()

In [ ]:
top_series = qc_resumen.sort_values("MesesObservados_QC", ascending=False).head(30)

plt.figure(figsize=(10, 8))
plt.barh(top_series["CodigoEstacion"], top_series["MesesObservados_QC"])
plt.title("Top 30 estaciones con más meses observados")
plt.xlabel("Meses observados")
plt.ylabel("Código estación")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
climatologia_mensual = (
    precipitacion
    .groupby("Mes")["Valor"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.plot(climatologia_mensual["Mes"], climatologia_mensual["Valor"], marker="o")
plt.title("Promedio mensual global de precipitación")
plt.xlabel("Mes")
plt.ylabel("Precipitación media mensual")
plt.xticks(range(1, 13))
plt.grid(True)
plt.show()

In [ ]:
registros_por_fecha = (
    precipitacion
    .groupby("Fecha")
    .agg(EstacionesConDato=("CodigoEstacion", "nunique"))
    .reset_index()
)

plt.figure(figsize=(12, 5))
plt.plot(registros_por_fecha["Fecha"], registros_por_fecha["EstacionesConDato"])
plt.title("Número de estaciones con dato por mes")
plt.xlabel("Fecha")
plt.ylabel("Estaciones con dato")
plt.grid(True)
plt.show()

## 12. Matriz de disponibilidad temporal

Esta matriz permite observar visualmente qué estaciones tienen datos en cada mes.

In [ ]:
disponibilidad = (
    precipitacion
    .assign(disponible=1)
    .pivot_table(
        index="CodigoEstacion",
        columns="Periodo",
        values="disponible",
        aggfunc="max"
    )
    .fillna(0)
)

# Para visualización, se muestran máximo 80 estaciones.
disponibilidad_muestra = disponibilidad.loc[
    qc_resumen.sort_values("MesesObservados_QC", ascending=False)["CodigoEstacion"].head(80)
]

plt.figure(figsize=(14, 8))
plt.imshow(disponibilidad_muestra, aspect="auto")
plt.title("Disponibilidad temporal por estación")
plt.xlabel("Meses")
plt.ylabel("Estaciones")
plt.yticks(range(len(disponibilidad_muestra.index)), disponibilidad_muestra.index)
plt.show()

## 13. Exportación de resultados QC

Se guardan la base mensual marcada con banderas de calidad, el resumen por estación y la capa espacial integrada.

In [ ]:
columnas_qc_mensual = [
    "CodigoEstacion",
    "NombreEstacion",
    "Fecha",
    "Periodo",
    "Mes",
    "Valor",
    "Unidad",
    "NivelAprobacion",
    "flag_valor_nulo",
    "flag_valor_negativo",
    "flag_atipico_iqr"
]

columnas_disponibles = [c for c in columnas_qc_mensual if c in precipitacion.columns]

RUTA_SALIDA_QC_MENSUAL.parent.mkdir(parents=True, exist_ok=True)

precipitacion[columnas_disponibles].to_csv(
    RUTA_SALIDA_QC_MENSUAL,
    index=False,
    encoding="utf-8-sig"
)

qc_resumen.to_csv(
    RUTA_SALIDA_QC_RESUMEN,
    index=False,
    encoding="utf-8-sig"
)

if estaciones_qc is not None:
    estaciones_qc.to_file(RUTA_SALIDA_QC_GPKG, driver="GPKG")

print("Archivo mensual QC:", RUTA_SALIDA_QC_MENSUAL.resolve())
print("Resumen estaciones QC:", RUTA_SALIDA_QC_RESUMEN.resolve())
if estaciones_qc is not None:
    print("Capa espacial QC:", RUTA_SALIDA_QC_GPKG.resolve())

## 14. Interpretación inicial

Al finalizar este notebook, conviene revisar:

1. Estaciones con baja completitud.
2. Estaciones con brechas largas.
3. Estaciones con valores negativos.
4. Estaciones con posibles atípicos.
5. Periodos históricos con baja densidad de estaciones.
6. Relación entre cobertura espacial, altitud y disponibilidad temporal.

La salida `qc_estaciones_precipitacion.gpkg` será especialmente útil para la siguiente fase, porque permitirá ver en mapa dónde están las estaciones de mayor y menor calidad temporal.